# Darcy equation: exercise 2

Let $\Omega=(0,1)^2$ with boundary $\partial \Omega$ and outward unit normal ${\nu}$. Given 
$k=I$ the matrix permeability and $g=(0, 1)$ a vector source term, we want to solve the following problem: find $({q}, p)$ such that
$$
\left\{
\begin{array}{ll}
\begin{array}{l} 
k^{-1} {q} + \nabla p = {g}\\
\nabla \cdot {q} = 0
\end{array}
&\text{in } \Omega
\end{array}
\right.
$$
with boundary conditions:
$$ p = 1 \text{ on } \partial_{top} \Omega \qquad p = 0 \text{ on } \partial_{bottom} \Omega \qquad \nu \cdot q = 0 \text{ on } \partial_{left} \Omega \cup \partial_{right} \Omega$$

This is the guided ("fill in the code") version of `ex2.ipynb` -- work through the cells in order, replacing each `# TODO` with your own implementation. Compare against `ex2.ipynb` once you're done, or if you get stuck.

First we import some of the standard modules.

In [ ]:
import numpy as np
import scipy.sparse as sps

import porepy as pp
import pygeon as pg

We create now the grid, in this example we consider a 2-dimensional grid.

In [ ]:
# TODO: create a 2d grid with pg.unit_grid(dim, mesh_size, as_mdg=False)
# (try mesh_size = 0.1 to start), then call sd.compute_geometry()


Let us declare the finite element spaces that we are going to use

In [ ]:
# TODO: declare the RT0 (for q) and PwConstants/P0 (for p) discretization
# objects under a key of your choice, e.g. key = "flow"
#
# TODO: build the degrees-of-freedom array
# dofs = np.array([rt0.ndof(sd), p0.ndof(sd)])


With the following code we set the data, in particular the permeability tensor and the boundary conditions. Since we need to identify each side of $\partial \Omega$ we need few steps.

In [ ]:
# TODO: set an isotropic, unitary permeability tensor with pp.SecondOrderTensor
# and pack it into a data dictionary with pp.initialize_data (see pg.SECOND_ORDER_TENSOR)
#
# TODO: identify the four sides of the domain from sd.face_centers (left/right/bottom/top)
#
# TODO: impose p = 1 on top and p = 0 on bottom as a NATURAL boundary condition for RT0
# (use rt0.assemble_nat_bc with a function returning the pressure value on the boundary),
# and nu.q = 0 (essential) on left/right
#
# TODO: build the vector source term g = (0, 1) by interpolating it with rt0.interpolate
# and multiplying by the RT0 mass matrix


Once the data are assigned to the grid, we construct the matrices. In particular, the linear system associated with the equation is given as
$$
\left(
\begin{array}{cc} 
A & -B^\top\\
B & 0
\end{array}
\right)
\left(
\begin{array}{c} 
q\\ 
p
\end{array}
\right)
=\left(
\begin{array}{c} 
p_{\partial} + g\\ 
0
\end{array}
\right)
$$<br>
where $p_{\partial}$ is the vector associated to the pressure boundary conditions. Once the matrix is created, we also construct the right-hand side containing the boundary conditions.

In [ ]:
# TODO: assemble the local matrices -- the RT0 mass matrix A (with data), the P0 mass
# matrix, and the divergence matrix B = mass_p0 @ rt0.assemble_diff_matrix(sd)
#
# TODO: assemble the saddle-point matrix spp with scipy.sparse.block_array
# ([[A, -B.T], [B, None]], format="csc")
#
# TODO: assemble the right-hand side rhs (length dofs.sum()), adding the boundary
# term and the vector source to the q-block (the first dofs[0] entries)


We need to solve the linear system, PyGeoN provides a framework for that. The actual imposition of essential boundary conditions (flux boundary conditions) might change the symmetry of the global system, the class `pg.LinearSystem` preserves this structure by internally eliminating these degrees of freedom. Once the problem is solved, we extract the two solutions $q$ and $p$.

In [ ]:
# TODO: build a pg.LinearSystem from spp and rhs, flag the essential boundary dofs
# with ls.flag_ess_bc(bc_ess, ...), then solve()
#
# TODO: split the solution vector into q and p, e.g. with
# idx = np.cumsum(dofs[:-1]); q, p = np.split(x, idx)


Since the computed $q$ is one value per facet of the grid, for visualization purposes we project the flux in each cell center as vector. We finally export the solution to be visualized by [ParaView](https://www.paraview.org/).

In [ ]:
# TODO: project q to cell centers with rt0.eval_at_cell_centers(sd), and evaluate p
# at cell centers with p0.eval_at_cell_centers(sd)
#
# TODO: export cell_p and cell_q with pp.Exporter(sd, "sol", folder_name="ex2").write_vtu(...)


In [ ]:
# Consistency check -- once your implementation is correct, this should pass
assert np.isclose(np.linalg.norm(cell_p), 8.917049238850113)
assert np.isclose(np.linalg.norm(cell_q), 0)